# Sistema de Gestão e Análise de Vendas

## Case analítico com dados públicos de comércio eletrônico brasileiro

Fonte: **Brazilian E-Commerce Public Dataset by Olist**.

Este notebook apresenta a análise de vendas a partir de uma base pública e anonimizada. A análise prioriza rastreabilidade, definição de granularidade, qualidade dos dados e interpretação de negócio.

## 1. Perguntas de negócio

1. Qual foi o volume de pedidos, itens e receita dos pedidos entregues?
2. Como a receita evoluiu ao longo do período observado?
3. Quais estados concentram a receita?
4. Quais categorias concentram maior receita?
5. Qual é o nível de recorrência dos clientes?
6. Existe concentração relevante entre vendedores?

As perguntas são respondidas somente com métricas suportadas pelas colunas disponíveis na fonte.

## 2. Regras analíticas

- A granularidade transacional é **um item de pedido** (`order_items`).
- Para receita realizada, entram itens de pedidos com `order_status = delivered`.
- `price` representa o valor dos itens; `freight_value` é analisado separadamente.
- `customer_unique_id` é usado para recorrência de clientes.
- Pagamentos e avaliações não são unidos diretamente à fato de itens.
- Pedidos não entregues permanecem disponíveis para análises operacionais, mas não compõem o KPI de receita realizada.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.transformation.olist import build_item_sales_fact, load_olist_tables
from src.analytics.sales_kpis import calculate_sales_kpis, revenue_by_category
from src.analytics.sales_analysis import (
    monthly_sales,
    sales_by_state,
    sales_by_seller,
    customer_purchase_frequency,
    repeat_customer_rate,
)

## 3. Ingestão e visão transacional

In [ ]:
tables = load_olist_tables(ROOT / 'dados' / 'raw')
sales = build_item_sales_fact(tables)
print(f'Tabelas carregadas: {len(tables)}')
print(f'Linhas na visão de itens: {len(sales):,}')
sales.head()

## 4. KPIs principais

Os KPIs abaixo são calculados somente para itens associados a pedidos entregues.

In [ ]:
kpis = calculate_sales_kpis(sales)
kpis

### Interpretação

A receita é apresentada como soma de `price`. O frete é mantido separado para evitar atribuir ao produto uma natureza de receita que não foi definida pela fonte.

## 5. Evolução temporal

In [ ]:
monthly = monthly_sales(sales)
monthly.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly['purchase_month_start'], monthly['revenue'], marker='o')
plt.title('Evolução mensal da receita dos pedidos entregues')
plt.xlabel('Mês de compra')
plt.ylabel('Receita dos itens (R$)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Cuidado de interpretação

Os primeiros meses possuem cobertura muito menor de pedidos entregues. Portanto, não devem ser usados isoladamente para inferir sazonalidade estrutural.

## 6. Concentração geográfica

In [ ]:
states = sales_by_state(sales)
states.head(10)

In [ ]:
top_states = states.head(10).sort_values('revenue')
plt.figure(figsize=(9, 6))
plt.barh(top_states['customer_state'], top_states['revenue'])
plt.title('Estados com maior receita dos pedidos entregues')
plt.xlabel('Receita dos itens (R$)')
plt.ylabel('UF do cliente')
plt.tight_layout()
plt.show()

### Concentração

A participação acumulada dos três principais estados pode ser usada como indicador simples de concentração geográfica, sem afirmar causalidade sobre o desempenho.

In [ ]:
top3_state_share = states.head(3)['revenue'].sum() / states['revenue'].sum()
print(f'Top 3 UFs por receita: {top3_state_share:.2%}')

## 7. Concentração por categoria

In [ ]:
categories = revenue_by_category(sales)
categories.head(10)

In [ ]:
top_categories = categories.head(10).sort_values('revenue')
plt.figure(figsize=(10, 6))
plt.barh(top_categories['product_category_name_english'], top_categories['revenue'])
plt.title('Categorias líderes em receita')
plt.xlabel('Receita dos itens (R$)')
plt.ylabel('Categoria')
plt.tight_layout()
plt.show()

In [ ]:
top5_category_share = categories.head(5)['revenue'].sum() / categories['revenue'].sum()
print(f'Top 5 categorias por receita: {top5_category_share:.2%}')

## 8. Vendedores e concentração

In [ ]:
sellers = sales_by_seller(sales)
sellers.head(10)

In [ ]:
top10_seller_share = sellers.head(10)['revenue'].sum() / sellers['revenue'].sum()
print(f'Top 10 vendedores por receita: {top10_seller_share:.2%}')

## 9. Recorrência de clientes

In [ ]:
customers = customer_purchase_frequency(sales)
repeat_rate = repeat_customer_rate(sales)
print(f'Taxa de clientes com mais de um pedido: {repeat_rate:.2%}')
customers.head(10)

## 10. Síntese executiva

Os resultados devem ser lidos em conjunto com a documentação de qualidade e modelagem. O objetivo da análise é identificar distribuição, concentração e comportamento de compra, sem extrapolar os dados para causalidade.

Os principais números oficiais e interpretações estão consolidados em `docs/insights-iniciais.md`.

## 11. Limitações

- A base representa uma amostra histórica de comércio eletrônico e não deve ser tratada como retrato atual do mercado brasileiro.
- Receita de produto é definida neste projeto como soma de `price`; margem e lucro não são calculados porque a fonte não fornece custo do produto.
- Pagamentos, avaliações e geolocalização exigem tratamento de granularidade específico antes de entrarem nos KPIs centrais.
- Diferenças entre estados, categorias ou vendedores não são interpretadas como causalidade.